<a href="https://colab.research.google.com/github/sayedmohamedscu/Vision-language-models-VLM/blob/main/medgemma_4b_it.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install --upgrade transformers bitsandbytes datasets evaluate peft trl scikit-learn kaggle

In [ ]:
!git clone https://huggingface.co/datasets/Manusinhh/medgemma_llava-med10k_dataset


Cloning into 'medgemma_llava-med10k_dataset'...
remote: Enumerating objects: 22, done.
remote: Counting objects: 100% (19/19), done.
remote: Compressing objects: 100% (19/19), done.
remote: Total 22 (delta 1), reused 0 (delta 0), pack-reused 3 (from 1)
Unpacking objects: 100% (22/22), 5.74 KiB | 1.15 MiB/s, done.
Filtering content: 100% (8/8), 3.17 GiB | 46.96 MiB/s, done.


In [ ]:
from datasets import load_from_disk

# Load the dataset
formatted_data = load_from_disk("medgemma_llava-med10k_dataset/val")
formatted_data

Dataset({
    features: ['image', 'messages'],
    num_rows: 500
})

In [ ]:
formatted_data[0]

{'image': <PIL.PngImagePlugin.PngImageFile image mode=RGB size=600x391>,
 'messages': [{'content': [{'text': None, 'type': 'image'},
    {'text': 'Analyze this medical image and provide step-by-step findings.',
     'type': 'text'}],
   'role': 'user'},
  {'content': [{'text': 'Analyzing Renal Parenchyma: A 2.2 cm clear cell carcinoma is identified on T2-weighted MRI of the left kidney.\nAnalyzing Tumor Appearance: The tumor appears “clear” based on the caption.\nAnalyzing Tumor Location: The lesion is located on the left kidney.\nFinal Answer: The patient is diagnosed with a clear cell carcinoma of the left kidney.\nWeb Search: Medical image mr clear',
     'type': 'text'}],
   'role': 'assistant'}]}

In [ ]:
import os
import ast
from PIL import Image
from datasets import load_dataset
from huggingface_hub import login
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
from peft import get_peft_model, LoraConfig

In [ ]:
login("")

In [ ]:
model_id = "google/medgemma-4b-it"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)
model = AutoModelForImageTextToText.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

processor = AutoProcessor.from_pretrained(model_id)
processor.tokenizer.padding_side = "right"

In [ ]:
# === Step 5: LoRA configuration ===
peft_config = LoraConfig(
    lora_alpha=16,
    lora_dropout=0.05,
    r=8,
    bias="none",
    target_modules="all-linear",
    task_type="CAUSAL_LM",
    modules_to_save=["lm_head", "embed_tokens"]
)
peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "o_proj", "k_proj", "v_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    task_type="CAUSAL_LM",
)

In [ ]:
model = get_peft_model(model, peft_config)

In [ ]:
def collate_fn(examples):
    prompts = []
    images_batch = []

    for ex in examples:
        prompt = processor.apply_chat_template(
            ex["messages"],
            tokenize=False,
            add_generation_prompt=False
        ).strip()
        prompts.append(prompt)
        images_batch.append(ex["image"])

    batch = processor(text=prompts, images=images_batch, return_tensors="pt", padding=True)

    # Print token IDs and their corresponding text
    # print(prompts)
    # print("\nToken IDs to Text Mapping:")
    for i, input_ids in enumerate(batch["input_ids"]):
        # print(f"\nSample {i+1}:")
        tokens = processor.tokenizer.convert_ids_to_tokens(input_ids)
        # for token_id, token in zip(input_ids, tokens):
        #     print(f"{token_id:>8} -> {token}")

    labels = batch["input_ids"].clone()

    # Masking logic (unchanged)
    image_token_id = [
        processor.tokenizer.convert_tokens_to_ids(
            processor.tokenizer.special_tokens_map["boi_token"]
        )
    ]
    labels[labels == processor.tokenizer.pad_token_id] = -100
    labels[labels == image_token_id] = -100
    labels[labels == 262144] = -100

    batch["labels"] = labels

    # print("\nFinal labels (with masking):")
    # print(labels)
    # print("x"*20)
    return batch

In [ ]:
import os

# Correct path for Kaggle
output_dir = "medgemma-qlora-finetune"
os.makedirs(output_dir, exist_ok=True)  # Creates dir if it doesn't exist

In [ ]:
training_args = SFTConfig(
    output_dir=output_dir,
    num_train_epochs=1,
    per_device_train_batch_size=1,        # optimized for low VRAM
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    optim="adamw_torch_fused",
    learning_rate=2e-4,
    bf16=True,
    max_grad_norm=0.3,
    warmup_ratio=0.03,
    lr_scheduler_type="linear",
    save_strategy="epoch",
    push_to_hub=True,
    logging_steps=0.1,
    eval_strategy="no",  # train-only dataset
    report_to="none",
    gradient_checkpointing_kwargs={"use_reentrant": False},
    dataset_kwargs={"skip_prepare_dataset": True},
    remove_unused_columns=False,
    label_names=["labels"],
    dataloader_num_workers=8,
    dataloader_pin_memory=True
)

In [ ]:
trainer = SFTTrainer(
        model=model,
        train_dataset=formatted_data_100 ,
        eval_dataset=None,
        data_collator=collate_fn,
        args=training_args,
        processing_class=processor,
        )

In [ ]:
trainer.train()


/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Step,Training Loss
7,26.008900
14,16.328400
21,11.971800
28,11.202300
35,10.079500
42,8.858300
49,9.293100


Step,Training Loss
7,26.008900
14,16.328400
21,11.971800
28,11.202300
35,10.079500
42,8.858300
49,9.293100
56,9.241800
63,8.267600


TrainOutput(global_step=63, training_loss=12.361299242292132, metrics={'train_runtime': 7502.517, 'train_samples_per_second': 0.067, 'train_steps_per_second': 0.008, 'total_flos': 4487603918296320.0, 'train_loss': 12.361299242292132})

In [ ]:
from datasets import load_from_disk

# Load the dataset
formatted_data = load_from_disk("medgemma_llava-med10k_dataset/val")
formatted_data

Dataset({
    features: ['image', 'messages'],
    num_rows: 500
})

inference

In [ ]:
formatted_data[0]['image']


In [ ]:
formatted_data[0]

{'image': <PIL.PngImagePlugin.PngImageFile image mode=RGB size=600x391>,
 'messages': [{'content': [{'text': None, 'type': 'image'},
    {'text': 'Analyze this medical image and provide step-by-step findings.',
     'type': 'text'}],
   'role': 'user'},
  {'content': [{'text': 'Analyzing Renal Parenchyma: A 2.2 cm clear cell carcinoma is identified on T2-weighted MRI of the left kidney.\nAnalyzing Tumor Appearance: The tumor appears “clear” based on the caption.\nAnalyzing Tumor Location: The lesion is located on the left kidney.\nFinal Answer: The patient is diagnosed with a clear cell carcinoma of the left kidney.\nWeb Search: Medical image mr clear',
     'type': 'text'}],
   'role': 'assistant'}]}

In [ ]:
# import torch
# from PIL import Image
# import requests
# from transformers import AutoModelForImageTextToText, AutoProcessor
# import os

# # Disable torch.compile to avoid the "Unsupported: generator" error
# torch._dynamo.config.disable = True

# # --- Configuration ---
# # Use the model from Hugging Face Hub or your local fine-tuned checkpoint
# MODEL_PATH = "/content/medgemma-qlora-finetune/checkpoint-25"  # or "kingabzpro/medgemma-brain-cancer" if using HF Hub

# # Automatically set device and data type
# DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# # Use bfloat16 if supported (on Ampere GPUs like A100), otherwise float16
# DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

# print(f"Using device: {DEVICE}")
# print(f"Using dtype: {DTYPE}")

# # --- Load Model & Processor ---
# model = AutoModelForImageTextToText.from_pretrained(
#     MODEL_PATH,
#     torch_dtype=DTYPE,
#     device_map="auto",  # Automatically handle model placement on devices
#     trust_remote_code=True  # Add this if needed for custom model code
# )
# processor = AutoProcessor.from_pretrained(MODEL_PATH, trust_remote_code=True)
# tokenizer = processor.tokenizer

# # --- Prepare Image and Prompt ---
# # Load your image
# image = Image.open("/content/x.jpg").convert("RGB")

# # The prompt for the model
# user_prompt = "Based on the provided brain MRI, what abnormalities do you observe?"

# # --- Create Chat Template ---
# # This formats the input correctly for the MedGemma model
# chat = [
#     {
#         "role": "user",
#         "content": [
#             {"type": "image"},
#             {"type": "text", "text": user_prompt}
#         ],
#     }
# ]
# formatted_prompt = processor.apply_chat_template(chat, add_generation_prompt=True, tokenize=False)

# # --- Run Inference ---
# # Process the text and image together
# inputs = processor(text=formatted_prompt, images=image, return_tensors="pt").to(DEVICE)

# # Move inputs to correct dtype if needed
# if hasattr(inputs, 'pixel_values') and inputs.pixel_values is not None:
#     inputs.pixel_values = inputs.pixel_values.to(dtype=DTYPE)

# input_ids_len = inputs["input_ids"].shape[-1]

# # Generate a response from the model with additional safeguards
# with torch.inference_mode():
#     try:
#         output_ids = model.generate(
#             **inputs,
#             max_new_tokens=200,
#             use_cache=True,
#             do_sample=False,  # Use greedy decoding for more stable results
#             pad_token_id=tokenizer.eos_token_id,  # Explicitly set pad token
#             temperature=0.7,  # Add temperature control
#             top_p=0.9,  # Add nucleus sampling
#         )
#     except Exception as e:
#         print(f"Error during generation: {e}")
#         print("Trying with simplified generation parameters...")
#         output_ids = model.generate(
#             input_ids=inputs["input_ids"],
#             pixel_values=inputs.get("pixel_values"),
#             max_new_tokens=200,
#             pad_token_id=tokenizer.eos_token_id,
#         )

# # Decode the generated tokens to text, skipping the prompt
# response = processor.decode(output_ids[0, input_ids_len:], skip_special_tokens=True)

# # --- Output ---
# print("\n📌 Model Prediction:")
# print(response)

Using device: cuda
Using dtype: torch.bfloat16


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



📌 Model Prediction:
Analyzing the CT scan reveals a focal lesion within the right lung apex, characterized by a heterogeneous appearance and surrounding ground-glass opacities.



In [ ]:
import torch
from PIL import Image
import requests
from transformers import AutoModelForImageTextToText, AutoProcessor
import os

# Disable torch.compile to avoid the "Unsupported: generator" error
torch._dynamo.config.disable = True

# --- Configuration ---
# Use the model from Hugging Face Hub or your local fine-tuned checkpoint
MODEL_PATH = "/content/medgemma-qlora-finetune/checkpoint-63"  # or "kingabzpro/medgemma-brain-cancer" if using HF Hub

# Automatically set device and data type
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# Use bfloat16 if supported (on Ampere GPUs like A100), otherwise float16
DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

print(f"Using device: {DEVICE}")
print(f"Using dtype: {DTYPE}")

# --- Load Model & Processor ---
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_PATH,
    torch_dtype=DTYPE,
    device_map="auto",  # Automatically handle model placement on devices
    trust_remote_code=True  # Add this if needed for custom model code
)
processor = AutoProcessor.from_pretrained(MODEL_PATH, trust_remote_code=True)
tokenizer = processor.tokenizer

In [ ]:


# --- Prepare Image and Prompt ---
# Load your image
# image = Image.open("/content/x.jpg").convert("RGB")
image =formatted_data[0]['image']

# The prompt for the model
user_prompt = "Analyze this medical image and provide step-by-step findings."

# --- Create Chat Template ---
# This formats the input correctly for the MedGemma model
chat = [
    {
        "role": "user",
        "content": [
            {"type": "image"},
            {"type": "text", "text": user_prompt}
        ],
    }
]
formatted_prompt = processor.apply_chat_template(chat, add_generation_prompt=True, tokenize=False)

# --- Run Inference ---
# Process the text and image together
inputs = processor(text=formatted_prompt, images=image, return_tensors="pt").to(DEVICE)

# Move inputs to correct dtype if needed
if hasattr(inputs, 'pixel_values') and inputs.pixel_values is not None:
    inputs.pixel_values = inputs.pixel_values.to(dtype=DTYPE)

input_ids_len = inputs["input_ids"].shape[-1]

# Generate a response from the model with additional safeguards
with torch.inference_mode():
    try:
        output_ids = model.generate(
            **inputs,
            max_new_tokens=200,
            use_cache=True,
            do_sample=False,  # Use greedy decoding for more stable results
            pad_token_id=tokenizer.eos_token_id,  # Explicitly set pad token
            temperature=0.7,  # Add temperature control
            top_p=0.9,  # Add nucleus sampling
        )
    except Exception as e:
        print(f"Error during generation: {e}")
        print("Trying with simplified generation parameters...")
        output_ids = model.generate(
            input_ids=inputs["input_ids"],
            pixel_values=inputs.get("pixel_values"),
            max_new_tokens=200,
            pad_token_id=tokenizer.eos_token_id,
        )

# Decode the generated tokens to text, skipping the prompt
response = processor.decode(output_ids[0, input_ids_len:], skip_special_tokens=True)

# --- Output ---
print("\n📌 Model Prediction:")
print(response)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



📌 Model Prediction:
Analyzing Renal Parenchyma: Axial T2-weighted MRI reveals a 1.5 cm hypointense lesion within the left kidney.
Analyzing Lesion Characteristics: The lesion is described as hypointense on T2-weighted imaging.
Analyzing Location: The lesion is located within the left kidney.
Analyzing Size: The lesion measures 1.5 cm in diameter.
Final Answer: The MRI findings are consistent with a renal cyst.
Web Search: MRI renal cyst


In [ ]:
print ("GT")
formatted_data[0]['messages'][1]['content'][0]['text']

GT


'Analyzing Renal Parenchyma: A 2.2 cm clear cell carcinoma is identified on T2-weighted MRI of the left kidney.\nAnalyzing Tumor Appearance: The tumor appears “clear” based on the caption.\nAnalyzing Tumor Location: The lesion is located on the left kidney.\nFinal Answer: The patient is diagnosed with a clear cell carcinoma of the left kidney.\nWeb Search: Medical image mr clear'